# Day 2 - Exercise 01: Prompt Templates and Chaining Workflows

## Real-World Scenario: Email Processing System

**Context:** You're building an email processing system for a company that:
1. Classifies incoming emails (support, sales, spam, general)
2. Extracts key information
3. Generates appropriate responses
4. Routes to the right department

**Learning Objectives:**
- Master different types of PromptTemplates
- Build multi-step chains with LCEL (LangChain Expression Language)
- Use output parsers for structured data
- Create branching workflows based on conditions

---

#### Setup

In [4]:
from dotenv import load_dotenv
load_dotenv()
import os

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    FewShotPromptTemplate,
    MessagesPlaceholder
)
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from pydantic import BaseModel, Field
from typing import List, Optional

llm = llm = ChatAnthropic(model="global.anthropic.claude-haiku-4-5-20251001-v1:0",api_key=os.environ.get("KEY"),base_url="https://llmgw-wp.tekstac.com", max_tokens=200)
print("Setup complete!")

Setup complete!


---
## Part 1: Understanding Prompt Templates

LangChain offers different template types for different use cases:

| Template Type | Use Case | Example |
|--------------|----------|--------|
| `PromptTemplate` | Simple text prompts | Summaries, translations |
| `ChatPromptTemplate` | Multi-turn conversations | Chatbots, agents |
| `FewShotPromptTemplate` | Learning from examples | Classification, formatting |

### 1.1 Basic PromptTemplate

In [5]:
# Simple PromptTemplate
simple_prompt = PromptTemplate.from_template(
    "Summarize the following email in one sentence:\n\n{email}"
)

# Test it
test_email = """
Hi,
I ordered a laptop last week (Order #12345) but haven't received any shipping confirmation.
Can you please check the status? I need it for a presentation next Monday.
Thanks,
John
"""

chain = simple_prompt | llm | StrOutputParser()
result = chain.invoke({"email": test_email})
print("Summary:", result)

Summary: John is requesting a shipping status update for a laptop he ordered (Order #12345) that he needs by next Monday for a presentation.


### 1.2 ChatPromptTemplate with Roles

In [6]:
# ChatPromptTemplate with system and human messages
email_classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an email classifier. Classify emails into one of these categories:
- SUPPORT: Technical issues, product problems, complaints
- SALES: Pricing inquiries, purchase requests, quotes
- SPAM: Promotional, unsolicited, suspicious
- GENERAL: Everything else

Respond with ONLY the category name."""),
    ("human", "Classify this email:\n\n{email}")
])

classifier_chain = email_classifier_prompt | llm | StrOutputParser()

# Test with different emails
emails = [
    "My laptop screen is flickering. Order #12345. Please help!",
    "Can you send me a quote for 50 units of Widget Pro?",
    "Congratulations! You've won $1,000,000! Click here to claim!",
    "Just wanted to say thanks for the great service last week."
]

print("Email Classifications:")
print("-" * 40)
for email in emails:
    category = classifier_chain.invoke({"email": email})
    print(f"{category}: {email[:50]}...")

Email Classifications:
----------------------------------------
SUPPORT: My laptop screen is flickering. Order #12345. Plea...
SALES: Can you send me a quote for 50 units of Widget Pro...
SPAM: Congratulations! You've won $1,000,000! Click here...
GENERAL: Just wanted to say thanks for the great service la...


### 1.3 FewShotPromptTemplate (Learning from Examples)

In [7]:
# Define examples for the model to learn from
examples = [
    {
        "email": "My order #5678 arrived damaged. The screen has a crack.",
        "extraction": '{{"order_id": "5678", "issue_type": "damaged_product", "product": "screen", "urgency": "high", "sentiment": "negative"}}'
    },
    {
        "email": "Thanks for the quick delivery! Love the new headphones.",
        "extraction": '{{"order_id": null, "issue_type": "positive_feedback", "product": "headphones", "urgency": "low", "sentiment": "positive"}}'
    },
    {
        "email": "I\'ve been waiting 2 weeks for order #9999. Where is it?!",
        "extraction": '{{"order_id": "9999", "issue_type": "delayed_shipping", "product": null, "urgency": "high", "sentiment": "negative"}}'
    }
]

# Create the example template - use input_variables to specify only the template variables
example_template = PromptTemplate(
    input_variables=["email", "extraction"],
    template="Email: {email}\nExtraction: {extraction}"
)

# Create the few-shot prompt
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Extract structured information from customer emails. Follow the exact JSON format shown in the examples.",
    suffix="Email: {email}\nExtraction:",
    input_variables=["email"]
)

extraction_chain = few_shot_prompt | llm | StrOutputParser()

# Test with a new email
new_email = "Order #1234 never arrived. I need a refund immediately!"
result = extraction_chain.invoke({"email": new_email})
print("Extracted Information:")
print(result)

Extracted Information:
```json
{"order_id": "1234", "issue_type": "missing_delivery", "product": null, "urgency": "high", "sentiment": "negative"}
```


---
## Part 2: Building Chains with LCEL

LangChain Expression Language (LCEL) uses the `|` operator to chain components.

```python
chain = prompt | llm | parser
```

This is equivalent to: `parser(llm(prompt(input)))`

### 2.1 Sequential Chains

In [8]:
# Step 1: Classify the email
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify emails as: SUPPORT, SALES, or GENERAL. Respond with only the category."),
    ("human", "{email}")
])

# Step 2: Generate response based on category
respond_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a customer service agent. Generate a response based on the email category.

Category: {category}
Guidelines:
- SUPPORT: Be empathetic, offer to help, ask for details if needed
- SALES: Be professional, provide information, offer to connect with sales team
- GENERAL: Be friendly and helpful"""),
    ("human", "Original email: {email}\n\nGenerate an appropriate response.")
])

# Build the chain
classify_chain = classify_prompt | llm | StrOutputParser()
respond_chain = respond_prompt | llm | StrOutputParser()

def process_email(email: str) -> dict:
    # Step 1: Classify
    category = classify_chain.invoke({"email": email})
    
    # Step 2: Generate response
    response = respond_chain.invoke({
        "email": email,
        "category": category
    })
    
    return {
        "category": category,
        "response": response
    }

# Test
test_email = "I'm interested in purchasing 100 licenses for my company. Can you provide bulk pricing?"
result = process_email(test_email)
print(f"Category: {result['category']}")
print(f"\nGenerated Response:\n{result['response']}")

Category: SALES

Generated Response:
**Subject: Bulk Licensing Inquiry – 100 Licenses**

Thank you for reaching out and showing interest in our product!

We're excited about the possibility of working with your company. Bulk licensing is something we absolutely can accommodate, and we offer competitive pricing for orders of that size.

To provide you with an accurate quote and discuss the best solution for your needs, I'd like to connect you with our sales team. They'll be able to:

- Provide detailed pricing for 100 licenses
- Discuss volume discounts and payment options
- Answer any questions about licensing terms and support
- Explore any customization or integration needs you might have

Could you please share a bit more information?
- What industry/department will be using the licenses?
- Do you have a timeline for implementation?
- Any specific features or integrations you require?

I'll make sure the right person reaches out to you shortly. We appreciate


### 2.2 Parallel Chains with RunnableParallel

In [9]:
# Process email in parallel: classify AND extract info at the same time

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify as SUPPORT, SALES, or GENERAL. Reply with only the category."),
    ("human", "{email}")
])

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract these fields from the email:
- sender_name: (or "Unknown")
- order_id: (or null)
- urgency: high/medium/low
- key_request: one sentence summary

Format as JSON."""),
    ("human", "{email}")
])

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze sentiment: POSITIVE, NEGATIVE, or NEUTRAL. Reply with only the sentiment."),
    ("human", "{email}")
])

# Create parallel chain
parallel_analysis = RunnableParallel(
    category=classify_prompt | llm | StrOutputParser(),
    extraction=extract_prompt | llm | StrOutputParser(),
    sentiment=sentiment_prompt | llm | StrOutputParser()
)

# Test
test_email = """
Hi, I'm Sarah from ABC Corp.
My order #7890 was supposed to arrive yesterday but tracking shows it's stuck.
I really need this for our product launch on Friday. Can someone help?
Thanks,
Sarah
"""

results = parallel_analysis.invoke({"email": test_email})

print("=" * 60)
print("PARALLEL EMAIL ANALYSIS")
print("=" * 60)
print(f"\nCategory: {results['category']}")
print(f"\nSentiment: {results['sentiment']}")
print(f"\nExtracted Info:\n{results['extraction']}")

PARALLEL EMAIL ANALYSIS

Category: SUPPORT

Sentiment: NEGATIVE

Extracted Info:
```json
{
  "sender_name": "Sarah",
  "order_id": "7890",
  "urgency": "high",
  "key_request": "Order #7890 is stuck in transit and needs to arrive by Friday for a product launch."
}
```


### 2.3 Conditional/Branching Chains

In [11]:
# Different response templates based on category

support_response_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a technical support specialist. Be empathetic and solution-oriented.
Always:
1. Acknowledge the issue
2. Ask clarifying questions if needed
3. Provide immediate steps they can try
4. Offer to escalate if needed"""),
    ("human", "Respond to this support email:\n{email}")
])

sales_response_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a sales representative. Be professional and informative.
Always:
1. Thank them for their interest
2. Provide relevant information
3. Offer to schedule a call
4. Include a clear next step"""),
    ("human", "Respond to this sales inquiry:\n{email}")
])

general_response_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer service agent. Be helpful and warm."),
    ("human", "Respond to this email:\n{email}")
])

# Create response chains
support_chain = support_response_prompt | llm | StrOutputParser()
sales_chain = sales_response_prompt | llm | StrOutputParser()
general_chain = general_response_prompt | llm | StrOutputParser()

def route_email(email: str) -> str:
    """Classify email and route to appropriate response chain."""
    # First classify
    category = classify_chain.invoke({"email": email}).strip().upper()
    
    # Route to appropriate chain
    if "SUPPORT" in category:
        response = support_chain.invoke({"email": email})
        dept = "Technical Support"
    elif "SALES" in category:
        response = sales_chain.invoke({"email": email})
        dept = "Sales Team"
    else:
        response = general_chain.invoke({"email": email})
        dept = "Customer Service"
    
    return f"[Routed to: {dept}]\n\n{response}"

# Test with different emails
print("=" * 60)
print("SUPPORT EMAIL")
print("=" * 60)
print(route_email("My software keeps crashing when I try to export. Error code: E-500"))

print("\n" + "=" * 60)
print("SALES EMAIL")
print("=" * 60)
print(route_email("We're evaluating your enterprise plan for our team of 50. What's included?"))

SUPPORT EMAIL
[Routed to: Technical Support]

**Subject: Re: Export Crash Issue - E-500 Error**

---

Hi there,

Thank you for reaching out, and I'm sorry you're experiencing crashes when exporting. I understand how frustrating that must be, and I'm here to help get this resolved quickly.

I see you're getting an **E-500 error**—this typically relates to file handling or system resource issues. Let me gather a bit more information so I can pinpoint the cause:

1. **What file format are you exporting to?** (PDF, Excel, CSV, etc.)
2. **What's the approximate size of the file you're trying to export?**
3. **Does this happen every time, or only with certain files?**
4. **Which version of our software are you currently running?**

**In the meantime, here are some immediate steps to try:**

- **Restart the application** an

SALES EMAIL
[Routed to: Sales Team]

Thank you for your interest in our enterprise plan! I'm excited to help you find the right solution for your team of 50.

**Our Enter

---
## Part 3: Structured Output with Parsers

In [12]:
# Define a Pydantic model for structured output
class EmailAnalysis(BaseModel):
    """Structured analysis of a customer email."""
    category: str = Field(description="SUPPORT, SALES, or GENERAL")
    urgency: str = Field(description="high, medium, or low")
    sentiment: str = Field(description="positive, negative, or neutral")
    order_id: Optional[str] = Field(description="Order ID if mentioned, otherwise null")
    customer_name: Optional[str] = Field(description="Customer name if provided")
    summary: str = Field(description="One sentence summary of the email")
    suggested_action: str = Field(description="Recommended next step")

# Create parser with the Pydantic model
json_parser = JsonOutputParser(pydantic_object=EmailAnalysis)

# Create prompt with format instructions
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze customer emails and extract structured information.

{format_instructions}

Be accurate and concise."""),
    ("human", "Analyze this email:\n\n{email}")
]).partial(format_instructions=json_parser.get_format_instructions())

# Build chain
structured_chain = structured_prompt | llm | json_parser

# Test
test_email = """
Subject: URGENT - Order #5555 Missing Items

Hi,
I'm Mike Thompson and I just received my order #5555 but 2 items are missing!
I ordered 5 widgets but only received 3. I need the other 2 ASAP for a client delivery tomorrow.
Please fix this immediately.
Mike
"""

analysis = structured_chain.invoke({"email": test_email})

print("=" * 60)
print("STRUCTURED EMAIL ANALYSIS")
print("=" * 60)
for key, value in analysis.items():
    print(f"{key}: {value}")

STRUCTURED EMAIL ANALYSIS
category: SUPPORT
urgency: high
sentiment: negative
order_id: 5555
customer_name: Mike Thompson
summary: Customer Mike Thompson received order #5555 with 2 missing widgets out of 5 ordered and urgently needs them for a client delivery tomorrow.
suggested_action: Immediately investigate order #5555 shipment records, locate the 2 missing widgets, and arrange expedited delivery or alternative resolution with priority handling given the time-sensitive client delivery deadline.


---
## Part 4: Complete Email Processing Pipeline

In [13]:
class EmailProcessor:
    """Complete email processing system with classification, extraction, and response generation."""
    
    def __init__(self, llm):
        self.llm = llm
        self._setup_chains()
    
    def _setup_chains(self):
        # Analysis chain (parallel)
        self.analysis_chain = RunnableParallel(
            category=ChatPromptTemplate.from_messages([
                ("system", "Classify as SUPPORT, SALES, SPAM, or GENERAL. Reply with only the category."),
                ("human", "{email}")
            ]) | self.llm | StrOutputParser(),
            
            urgency=ChatPromptTemplate.from_messages([
                ("system", "Rate urgency as HIGH, MEDIUM, or LOW. Reply with only the rating."),
                ("human", "{email}")
            ]) | self.llm | StrOutputParser(),
            
            sentiment=ChatPromptTemplate.from_messages([
                ("system", "Rate sentiment as POSITIVE, NEGATIVE, or NEUTRAL. Reply with only the rating."),
                ("human", "{email}")
            ]) | self.llm | StrOutputParser()
        )
        
        # Response templates
        self.response_templates = {
            "SUPPORT": ChatPromptTemplate.from_messages([
                ("system", """You are a support specialist. Write a helpful, empathetic response.
Urgency level: {urgency}
Customer sentiment: {sentiment}
Adapt your tone accordingly."""),
                ("human", "Original email: {email}\n\nWrite a response.")
            ]),
            "SALES": ChatPromptTemplate.from_messages([
                ("system", "You are a sales representative. Write a professional, informative response."),
                ("human", "Original email: {email}\n\nWrite a response.")
            ]),
            "GENERAL": ChatPromptTemplate.from_messages([
                ("system", "You are a customer service agent. Write a friendly, helpful response."),
                ("human", "Original email: {email}\n\nWrite a response.")
            ]),
            "SPAM": None  # No response for spam
        }
    
    def process(self, email: str) -> dict:
        """Process an email through the complete pipeline."""
        # Step 1: Analyze in parallel
        analysis = self.analysis_chain.invoke({"email": email})
        category = analysis["category"].strip().upper()
        
        # Step 2: Route and generate response (if not spam)
        if "SPAM" in category:
            response = "[SPAM DETECTED - No response generated]"
            routing = "Spam Folder"
        else:
            # Determine routing
            if "SUPPORT" in category:
                routing = "Technical Support Queue"
                template = self.response_templates["SUPPORT"]
            elif "SALES" in category:
                routing = "Sales Team"
                template = self.response_templates["SALES"]
            else:
                routing = "General Inbox"
                template = self.response_templates["GENERAL"]
            
            # Generate response
            response_chain = template | self.llm | StrOutputParser()
            response = response_chain.invoke({
                "email": email,
                "urgency": analysis["urgency"],
                "sentiment": analysis["sentiment"]
            })
        
        return {
            "category": category,
            "urgency": analysis["urgency"].strip(),
            "sentiment": analysis["sentiment"].strip(),
            "routing": routing,
            "response": response
        }

# Create processor
processor = EmailProcessor(llm)
print("Email processor ready!")

Email processor ready!


In [14]:
# Test the complete pipeline
test_emails = [
    {
        "subject": "Support Request",
        "body": """Hi, my account has been locked for 3 days and I can't access my files.
This is affecting my work. Please help urgently! - David"""
    },
    {
        "subject": "Enterprise Inquiry",
        "body": """Hello, our company is interested in your enterprise solution for 200+ users.
Could you provide pricing and a demo? Best regards, Jennifer (CTO, TechCorp)"""
    },
    {
        "subject": "Free Money!!!",
        "body": """YOU HAVE WON $10,000,000!!! Click here NOW to claim your prize!!!
Limited time offer! Send your bank details immediately!"""
    }
]

for email in test_emails:
    print("\n" + "=" * 60)
    print(f"PROCESSING: {email['subject']}")
    print("=" * 60)
    
    result = processor.process(email["body"])
    
    print(f"\n Analysis:")
    print(f"   Category: {result['category']}")
    print(f"   Urgency: {result['urgency']}")
    print(f"   Sentiment: {result['sentiment']}")
    print(f"   Routing: {result['routing']}")
    print(f"\n Generated Response:")
    print(f"   {result['response'][:300]}..." if len(result['response']) > 300 else f"   {result['response']}")


PROCESSING: Support Request

 Analysis:
   Category: SUPPORT
   Urgency: HIGH
   Sentiment: NEGATIVE
   Routing: Technical Support Queue

 Generated Response:
   # Response to David

Subject: URGENT: Your Account Access - We're Here to Help

Hi David,

Thank you for reaching out, and I sincerely apologize that your account has been locked for three days. I completely understand how frustrating and disruptive this must be for your work, and I want to make thi...

PROCESSING: Enterprise Inquiry

 Analysis:
   Category: SALES
   Urgency: HIGH
   Sentiment: NEUTRAL
   Routing: Sales Team

 Generated Response:
   **Subject: RE: Enterprise Solution Inquiry – TechCorp**

Dear Jennifer,

Thank you for your interest in our enterprise solution. We're excited about the opportunity to support TechCorp's growth with a platform designed for organizations of your scale.

**Next Steps:**

I'd be happy to provide you wi...

PROCESSING: Free Money!!!

 Analysis:
   Category: SPAM
   Urgency: LOW
 

---
##  Your Turn: Practice Exercises

### Exercise 1.1: Create a Translation Chain

Build a chain that:
1. Detects the language of input text
2. Translates to English if not English
3. Summarizes the content

In [15]:
# Step 1: Language detection prompt
detect_prompt = ChatPromptTemplate.from_messages([
    ("system", "Detect the language of the text. Reply with only the language name (e.g., 'English')"),
    ("human", "{text}")
])
 
# Step 2: Translation prompt
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following {source_language} text to English. If already English, return it as is."),
    ("human", "{text}")
])
 
# Step 3: Summary prompt
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one short, concise sentence."),
    ("human", "{text}")
])
 
# Build the complete chain using LCEL
# - We use RunnablePassthrough.assign to inject the detected language into the input dict for Step 2.
# - We then format the translated output into a new dict key {"text": ...} for the summary step.
complete_chain = (
    RunnablePassthrough.assign(
        source_language=detect_prompt | llm | StrOutputParser()
    )
    | translate_prompt
    | llm
    | StrOutputParser()
    | (lambda translated_text: {"text": translated_text})
    | summary_prompt
    | llm
    | StrOutputParser()
)
 
# Test with the provided sample text
test_text = "Bonjour, je voudrais réserver une table pour deux personnes ce soir à 20 heures."
result = complete_chain.invoke({"text": test_text})
 
print("Summary Output:")
print(result)

Summary Output:
I appreciate your interest, but I should clarify that I'm an AI assistant—I don't have the ability to make restaurant reservations. 

To book a table, you could:
- **Call the restaurant directly**
- **Use reservation apps** like OpenTable, Resy, or Yelp
- **Visit the restaurant's website** if they have online booking
- **Visit in person** to make a reservation

Would you like help finding a restaurant or tips on how to make a reservation?


### Exercise 1.2: Build a Product Review Analyzer

Create a parallel chain that analyzes product reviews for:
- Overall sentiment (positive/negative/neutral)
- Key pros mentioned
- Key cons mentioned
- Star rating prediction (1-5)

In [ ]:
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser
import json
 
# TODO: Build a product review analyzer
 
sample_review = """
I bought this laptop 3 months ago. The screen quality is amazing and the battery lasts all day.
However, the keyboard feels a bit mushy and it runs hot when gaming. For the price, it's decent
but not exceptional. Would recommend for casual users but not power users.
"""
 
# 1. Create prompts for each analysis dimension
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the overall sentiment of the review. Respond with exactly one word: Positive, Negative, or Neutral."),
    ("human", "{review}")
])
 
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List the key pros mentioned in this review as concise bullet points."),
    ("human", "{review}")
])
 
cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List the key cons mentioned in this review as concise bullet points."),
    ("human", "{review}")
])
 
rating_prompt = ChatPromptTemplate.from_messages([
    ("system", "Predict the star rating for this product review on a scale of 1 to 5. Respond with only the number."),
    ("human", "{review}")
])
 
#. Create parallel chain
# 1. Explicitly wrap the dictionary with RunnableParallel
parallel_analyzer = RunnableParallel({
    "overall_sentiment": sentiment_prompt | llm | StrOutputParser(),
    "key_pros": pros_prompt | llm | StrOutputParser(),
    "key_cons": cons_prompt | llm | StrOutputParser(),
    "predicted_rating": rating_prompt | llm | StrOutputParser()
})
# 2. Test with the sample review (this will now work smoothly!)
output = parallel_analyzer.invoke({"review": sample_review})
 
# 3. Print the results nicely
print(json.dumps(output, indent=4))
 

{
    "overall_sentiment": "Neutral",
    "key_pros": "# Key Pros from the Review:\n\n- Amazing screen quality\n- Battery lasts all day\n- Decent value for the price",
    "key_cons": "# Key Cons from Review\n\n- Mushy keyboard feel\n- Runs hot during gaming\n- Not exceptional for the price\n- Not suitable for power users",
    "predicted_rating": "3"
}


### Exercise 1.3: Create a Document Processing Pipeline

Build a pipeline for processing legal documents:
1. Extract key dates and parties involved
2. Identify the document type (contract, agreement, notice, etc.)
3. Summarize main obligations
4. Flag potential risks or concerns

In [19]:
sample_document = """
SERVICE AGREEMENT

This Agreement is entered into as of January 15, 2024, between:
- TechCorp Inc. ("Service Provider"), located at 123 Tech Street, San Francisco, CA
- ClientCo LLC ("Client"), located at 456 Business Ave, New York, NY

TERMS:
1. Service Provider agrees to deliver software development services for a period of 12 months.
2. Client agrees to pay $50,000 per month, due on the 1st of each month.
3. Late payments will incur a 5% penalty fee.
4. Either party may terminate with 30 days written notice.
5. All intellectual property created shall belong to the Client.
6. Service Provider shall maintain confidentiality for 5 years after termination.

Signed by both parties on the date first written above.
"""

# YOUR CODE HERE
 
# 1. Define specific prompt templates for each processing dimension
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract all key dates and parties involved from the legal document. Present them clearly."),
    ("human", "{document}")
])
 
doc_type_prompt = ChatPromptTemplate.from_messages([
    ("system", "Identify the document type (e.g., contract, agreement, notice, NDA). Give a one-sentence reason."),
    ("human", "{document}")
])
 
obligations_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the main obligations, deliverables, and payment terms specified in this document as bullet points."),
    ("human", "{document}")
])
 
risks_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the document and flag any potential risks, liabilities, or strict penalty fees for either party."),
    ("human", "{document}")
])
 
# 2. Build the parallel document processing pipeline
# We wrap the dictionary in RunnableParallel to avoid the dict attribute error seen in previous exercises!
legal_processing_pipeline = RunnableParallel({
    "extracted_details": extract_prompt | llm | StrOutputParser(),
    "document_type": doc_type_prompt | llm | StrOutputParser(),
    "main_obligations": obligations_prompt | llm | StrOutputParser(),
    "potential_risks": risks_prompt | llm | StrOutputParser()
})
 
# 3. Invoke the pipeline with the provided sample_document
output = legal_processing_pipeline.invoke({"document": sample_document})
 
# 4. Print the comprehensive legal analysis nicely formatted
print(json.dumps(output, indent=4))
 

{
    "extracted_details": "# KEY DATES AND PARTIES\n\n## **PARTIES INVOLVED**\n\n| Party | Role | Location |\n|-------|------|----------|\n| **TechCorp Inc.** | Service Provider | 123 Tech Street, San Francisco, CA |\n| **ClientCo LLC** | Client | 456 Business Ave, New York, NY |\n\n## **KEY DATES**\n\n| Date | Significance |\n|------|--------------|\n| **January 15, 2024** | Agreement Execution Date (Effective Date) |\n| **1st of each month** | Payment Due Date (recurring) |\n| **12 months** | Service Contract Duration |\n| **30 days** | Notice Period for Termination |\n| **5 years** | Confidentiality Obligation Period (post-termination) |\n\n## **CRITICAL PAYMENT DETAILS**\n- **Monthly",
    "document_type": "**Document Type:** Service Agreement\n\n**Reason:** This document outlines the terms and conditions for the provision of software development services between two parties, including payment obligations, deliverables, and contractual responsibilities, which are the defining char

---
##  Key Takeaways

1. **PromptTemplate** — Simple text templates with variables
2. **ChatPromptTemplate** — Multi-message templates with roles (system, human, AI)
3. **FewShotPromptTemplate** — Learning from examples for consistent formatting
4. **LCEL (`|` operator)** — Chains components: prompt → llm → parser
5. **RunnableParallel** — Execute multiple chains simultaneously
6. **Conditional routing** — Branch based on intermediate results
7. **Output parsers** — Extract structured data (JSON, Pydantic models)

---

**Next Exercise →** Tool Integration and Function Calling